# FastDOCX — Demo Notebook

Builds the native shared library, generates several `.docx` files using FastDOCX, and summarises the results in an artifacts table.


## 1 — Build the native shared library


In [22]:
import platform
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

# Resolve the RID and output path for this platform
_RID_MAP = {
    ("Linux", "x86_64"): ("linux-x64", "FastDocx.Native.so"),
    ("Linux", "aarch64"): ("linux-arm64", "FastDocx.Native.so"),
    ("Windows", "AMD64"): ("win-x64", "FastDocx.Native.dll"),
    ("Darwin", "arm64"): ("osx-arm64", "FastDocx.Native.dylib"),
}
_system, _machine = platform.system(), platform.machine()
_rid, _lib_name = _RID_MAP.get((_system, _machine), (None, None))

if not _rid or not _lib_name:
    raise RuntimeError(f"Unsupported platform: {_system}/{_machine}")

if _rid is None:
    print(f"⚠  Unsupported platform {_system}/{_machine} — skipping native build.")
    NATIVE_AVAILABLE = False
else:
    _lib_dir = REPO_ROOT / "fastdocx" / "_libs" / _rid
    _lib_path = _lib_dir / _lib_name

    if _lib_path.exists():
        print(f"✔  Native binary already present: {_lib_path}")
        NATIVE_AVAILABLE = True
    else:
        print(f"Building native library for {_rid} …")
        result = subprocess.run(
            [
                "dotnet",
                "publish",
                str(REPO_ROOT / "native" / "FastDocx.Native"),
                "-r",
                _rid,
                "-c",
                "Release",
                "-p:PublishAot=true",
                "-p:NativeLib=Shared",
                "-o",
                str(_lib_dir),
            ],
            capture_output=True,
            text=True,
        )
        print(result.stdout[-3000:] if result.stdout else "")
        if result.returncode != 0:
            print("STDERR:", result.stderr[-2000:])
            print("⚠  Native build failed — document operations will raise RuntimeError.")
            NATIVE_AVAILABLE = False
        else:
            print(f"✔  Built → {_lib_path}")
            NATIVE_AVAILABLE = _lib_path.exists()


✔  Native binary already present: /home/jrhol/programming/projects/fastdocx/fastdocx/_libs/linux-arm64/FastDocx.Native.so


## 2 — Create documents


In [23]:
import datetime, os
from fastdocx import Document

ARTIFACTS = REPO_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

_log: list[dict] = []  # filled in by each helper below


def _record(path: Path) -> None:
    stat = path.stat()
    _log.append(
        {
            "file": path.name,
            "size_kb": round(stat.st_size / 1024, 1),
            "modified": datetime.datetime.fromtimestamp(stat.st_mtime).strftime("%H:%M:%S"),
            "path": str(path.relative_to(REPO_ROOT)),
        }
    )
    print(f"  ✔  {path.name}  ({stat.st_size:,} bytes)")


In [24]:
### 2a — Quickstart (README example)
print("Creating quickstart.docx …")
doc = Document()
doc.add_heading("Project Report", level=1)
doc.add_paragraph("This report summarises Q1 findings.", bold=True)

table = doc.add_table(rows=3, cols=2)
table[0, 0].text = "Region"
table[0, 1].text = "Revenue"
table[1, 0].text = "North"
table[1, 1].text = "$1.2M"
table[2, 0].text = "South"
table[2, 1].text = "$0.9M"

out = ARTIFACTS / "quickstart.docx"
doc.save(str(out))
_record(out)


Creating quickstart.docx …
  ✔  quickstart.docx  (1,921 bytes)


In [25]:
### 2b — Headings at every level
print("Creating headings.docx …")
doc = Document()
doc.add_heading("Document Title", level=1)
doc.add_paragraph(
    "FastDOCX supports headings at all six Word heading levels.",
    style="Normal",
)
for lvl in range(1, 7):
    doc.add_heading(f"Heading {lvl}", level=lvl)
    doc.add_paragraph(f"Body text beneath heading level {lvl}.")

out = ARTIFACTS / "headings.docx"
doc.save(str(out))
_record(out)


Creating headings.docx …
  ✔  headings.docx  (1,887 bytes)


In [26]:
### 2c — Rich text formatting
print("Creating formatting.docx …")
doc = Document()
doc.add_heading("Text Formatting Showcase", level=1)
doc.add_paragraph("Plain paragraph — no extra formatting.")
doc.add_paragraph("Bold text.", bold=True)
doc.add_paragraph("Italic text.", italic=True)
doc.add_paragraph("Bold and italic.", bold=True, italic=True)
doc.add_paragraph("Large text (18 pt).", font_size=18)
doc.add_paragraph("Small text (8 pt).", font_size=8)
doc.add_paragraph("Bold large text (16 pt).", bold=True, font_size=16)

out = ARTIFACTS / "formatting.docx"
doc.save(str(out))
_record(out)


Creating formatting.docx …
  ✔  formatting.docx  (1,898 bytes)


In [27]:
### 2d — Large table (pre-populated via `data=`)
import random, string

random.seed(42)

ROWS, COLS = 10, 5
headers = ["ID", "Name", "Department", "Score", "Grade"]
grades = ["A", "B", "C", "D", "F"]
depts = ["Engineering", "Marketing", "Finance", "HR", "Legal"]

data = [headers]
for i in range(1, ROWS):
    name = "".join(random.choices(string.ascii_uppercase, k=6)).capitalize()
    score = random.randint(40, 100)
    data.append(
        [
            str(i),
            name,
            random.choice(depts),
            str(score),
            grades[min((100 - score) // 12, 4)],
        ]
    )

print("Creating large_table.docx …")
doc = Document()
doc.add_heading("Employee Scores", level=1)
doc.add_table(rows=ROWS, cols=COLS, data=data)

out = ARTIFACTS / "large_table.docx"
doc.save(str(out))
_record(out)


Creating large_table.docx …
  ✔  large_table.docx  (2,058 bytes)


## 3 — Artifacts table


In [28]:
from IPython.display import HTML

_col_widths = {"file": "180px", "size_kb": "80px", "modified": "90px", "path": "auto"}
_headers = {"file": "File", "size_kb": "Size (KB)", "modified": "Saved at", "path": "Path"}


def _table_html(rows: list[dict]) -> str:
    th = "".join(
        f'<th style="text-align:left;padding:6px 12px;background:#2d2d2d;'
        f'color:#e0e0e0;width:{_col_widths[k]}">{_headers[k]}</th>'
        for k in _headers
    )
    body = ""
    for i, row in enumerate(rows):
        bg = "#1e1e1e" if i % 2 == 0 else "#252525"
        tds = "".join(
            f'<td style="padding:5px 12px;color:#d4d4d4;font-family:monospace">{row[k]}</td>'
            for k in _headers
        )
        body += f'<tr style="background:{bg}">{tds}</tr>'
    return (
        '<table style="border-collapse:collapse;width:100%;font-size:13px">'
        f"<thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>"
    )


total_kb = sum(r["size_kb"] for r in _log)
print(f"Generated {len(_log)} artifact(s) — {total_kb:.1f} KB total\n")
HTML(_table_html(_log))


Generated 4 artifact(s) — 7.6 KB total



File,Size (KB),Saved at,Path
quickstart.docx,1.9,23:31:11,artifacts/quickstart.docx
headings.docx,1.8,23:31:11,artifacts/headings.docx
formatting.docx,1.9,23:31:11,artifacts/formatting.docx
large_table.docx,2.0,23:31:11,artifacts/large_table.docx
